In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam, SGD, lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from UNET_LIB.InceptionUnet import InceptionUNet
from Brats_Dataset import Brats_Dataset
from torch.utils.data import Dataset, Subset, DataLoader
from torch.utils.tensorboard import SummaryWriter
train_total=10240

group_spec={
    'perfe':160,
    'poly+':213,
    'poly-':572,
    'rough':868,
    'bbox_msk':1562,
    'sam_box':1562,
    'point_msk':1615
}

mult_spec={
    'perfe':[1,2,4,8,16,32,64],
    'poly+':[1,2,4,8,16,32,train_total/group_spec['poly+']],
    'poly-':[1,2,4,8,16,train_total/group_spec['poly-']],
    'rough':[1,2,4,8,train_total/group_spec['rough']],
    'bbox_msk':[1,2,4,train_total/group_spec['bbox_msk']],
    'sam_box':[1,2,4,train_total/group_spec['sam_box']],
    'point_msk':[1,2,4,train_total/group_spec['point_msk']]
}

pretrained = 'pb'
assert pretrained in ['pb','fb','ub']


img_preprocess = transforms.Compose([    

    transforms.RandomEqualize(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0,1),

])


msk_preprocess = transforms.Compose([

])

test_loader = DataLoader(Brats_Dataset('/data/Brats20/','test','perfe', 'f_', \
                                        test_preprocess, msk_preprocess,flip=False, rot=False,crop=False),\
                         batch_size=64, shuffle=False, num_workers=32,pin_memory=True)
train_val0 = Brats_Dataset('/data/Brats20/','trainval','perfe','f_',\
                           img_preprocess, msk_preprocess,flip=False, rot=False, crop=False)

val_data = Subset(Brats_Dataset('/data/Brats20/','trainval','perfe','f_',\
                                img_preprocess, msk_preprocess,flip=False, rot=False, crop=False),\
                  list(range(train_total, len(train_val0))))

val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=32,pin_memory=True)

num_classes = 2
max_iter = 12001
interval = 300
loss_fn = nn.CrossEntropyLoss(ignore_index=255) 

In [2]:
for label_type in ['point_msk']: #['perfe','poly+','poly-','rough','bbox_msk']:
    group_size = group_spec[label_type]
    DATA = Brats_Dataset('./Brats20/','trainval',label_type,'f_', img_preprocess, msk_preprocess,crop=False)
    for multiplicity in reversed(mult_spec[label_type]):
        
        train_subset = Subset(DATA, list(range(round(multiplicity*group_size))))
        train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True, num_workers=32,pin_memory=True)
        
        iteration = 1
        min_loss = np.inf
        
        model = InceptionUNet(1, n_classes=num_classes)
        optimizer = Adam(model.parameters(),lr=1e-3,eps=0.1, weight_decay=1e-6)
        model = nn.DataParallel(model).cuda()
        model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
       
        train_iter = iter(train_loader)
        model.train()
        train_loss = 0
        writer = SummaryWriter()
        writer.add_text('Variant', model_name)
        
        while iteration < max_iter:            
            
            optimizer.zero_grad()
            try:
                X,y = next(train_iter)
            except:
                train_iter = iter(train_loader)
                X,y = next(train_iter)
                
            yhat = model(X.contiguous().cuda())
            loss = loss_fn(yhat,y.cuda())
            loss.backward()
            optimizer.step()
            iteration += 1
            train_loss += loss.item()
            
            if iteration%interval == 0:
                train_loss /= interval
                writer.add_scalar('Loss/train', train_loss, iteration)
                train_loss = 0.
                
                model.eval()
                with torch.no_grad():
                    for eval_loader, set_name in zip([val_loader,test_loader],['Val','Test']):
                        
                        class_intersect = np.zeros((num_classes,),dtype='float')
                        class_union= np.zeros((num_classes,),dtype='float')
                        for idx,(X,y) in enumerate(eval_loader):
                            y = y.cuda().contiguous().flatten()
                            yhat = model(X.contiguous().cuda())
                            yhat_lab = torch.argmax(yhat, dim=1).flatten()

                            for j in range(num_classes):

                                y_bi = y == j
                                yhat_bi = yhat_lab == j
                                I = ((y_bi * yhat_bi).sum()).item()
                                U = (y_bi.sum() + yhat_bi.sum() - I).item()
                                assert I <= U
                                class_intersect[j] += I
                                class_union[j] += U

                        IOUs = class_intersect/class_union
                        for cls_id in range(len(IOUs)):
                            writer.add_scalar(f'{set_name}/cls{cls_id}', IOUs[cls_id],iteration)

                        if set_name == 'Val' and (-IOUs[1] < min_loss):
                            min_loss = -IOUs[1]
                            to_save ={'model_state_dict': model.state_dict()}

                            if pretrained=='fb':
                                to_save['scheduler_state_dict']= scheduler.state_dict()

                            torch.save(to_save, f'/data/model_checkpoints/{model_name}.pth')  
            model.train()

In [4]:
pretrained='pb'
performance={}
num_classes=2
test_preprocess = transforms.Compose([    

    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_loader = DataLoader( Brats_Dataset('./Brats20/','test','perfe',
                                              'f_', test_preprocess, msk_preprocess,flip=False, rot=False, crop=False), 
                         batch_size=64, shuffle=False, num_workers=8)

model = InceptionUNet(1, n_classes=num_classes)
model = nn.DataParallel(model).cuda()

for label_type in mult_spec.keys():
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        model_name = f'XUNet_B{label_type}_m{round(multiplicity)}_{pretrained}' 
        print(model_name)
        checkpoint = torch.load(f'./new_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
                y = y.flatten()

                yhat = model(X.contiguous().cuda())
                yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
                skip_id = np.argwhere(y == 255)
                yhat_lab[skip_id] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
                    
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Brats_Inter-Union_{pretrained}2', performance)


XUNet_Bperfe_m1_pb
Working on label_type=perfe:1
XUNet_Bperfe_m2_pb
Working on label_type=perfe:2
XUNet_Bperfe_m4_pb
Working on label_type=perfe:4
XUNet_Bperfe_m8_pb
Working on label_type=perfe:8
XUNet_Bperfe_m16_pb
Working on label_type=perfe:16
XUNet_Bperfe_m32_pb
Working on label_type=perfe:32
XUNet_Bperfe_m64_pb
Working on label_type=perfe:64
XUNet_Bpoly+_m1_pb
Working on label_type=poly+:1
XUNet_Bpoly+_m2_pb
Working on label_type=poly+:2
XUNet_Bpoly+_m4_pb
Working on label_type=poly+:4
XUNet_Bpoly+_m8_pb
Working on label_type=poly+:8
XUNet_Bpoly+_m16_pb
Working on label_type=poly+:16
XUNet_Bpoly+_m32_pb
Working on label_type=poly+:32
XUNet_Bpoly+_m48_pb
Working on label_type=poly+:48
XUNet_Bpoly-_m1_pb
Working on label_type=poly-:1
XUNet_Bpoly-_m2_pb
Working on label_type=poly-:2
XUNet_Bpoly-_m4_pb
Working on label_type=poly-:4
XUNet_Bpoly-_m8_pb
Working on label_type=poly-:8
XUNet_Bpoly-_m16_pb
Working on label_type=poly-:16
XUNet_Bpoly-_m18_pb
Working on label_type=poly-:18
XUNe

In [9]:
####some old code --abandoned
for i in range(len(multiplicity_list)): 
    print(f'multiplicity:{multiplicity_list[i]}: ', np.mean(class_intersect[i]/class_union[i]),\
          np.std(class_intersect[i]/class_union[i]),
          np.mean(2*class_intersect[i]/(class_intersect[i]+class_union[i])),
          np.std(2*class_intersect[i]/(class_intersect[i]+class_union[i])))

multiplicity:1:  0.13222235168842367 0.07390113808500424 0.2262246641991838 0.11276045202718044
multiplicity:2:  0.14710304129698945 0.06883423321089506 0.25004757290650065 0.10740431791076314
multiplicity:4:  0.15079518159805505 0.07190442744217519 0.2553373460144782 0.10831801359687405
multiplicity:8:  0.15772297539210886 0.0775480918617959 0.26475788700436587 0.1157308321814092


In [3]:
#snipping result

label_type='perfe'
test_loader = DataLoader(VOCSeg_Dataset('/data/VOC2012/','val','perfe',img_preprocess,msk_preprocess,crop=True,crop_size=500), batch_size=16, shuffle=False)

for multiplicity in mult_spec[label_type]:
        
    model = UNet(in_channels = 1, num_classes=num_classes)
    model = nn.DataParallel(model).cuda()
        
    model_name = f'DLab_V{label_type}_m{round(multiplicity)}_{pretrained}' 
    try:
        checkpoint = torch.load(f'/data/model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print('Loaded', model_name)
    except:
        print(f'Error loading {model_name}')
        assert False

    model.eval()
    with torch.no_grad():    
        class_intersect = np.zeros((num_classes,),dtype='float')
        class_union= np.zeros((num_classes,),dtype='float')

        for idx,(X,y) in enumerate(test_loader):
            y = y.flatten()

            yhat = model(X.contiguous().cuda())['out']
            yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
            skip_id = np.argwhere(y == 255)
            yhat_lab[skip_id] = 255

            for j in range(num_classes):

                y_bi = y == j
                yhat_bi = yhat_lab == j
                I = (y_bi * yhat_bi).sum()
                U = y_bi.sum() + yhat_bi.sum() - I
                assert I <= U
                class_intersect[j] += I
                class_union[j] += U

        IOUs = class_intersect/class_union
        val_loss=-np.mean(IOUs)
        print('IOU', -val_loss)
            

Loaded DLab_Vperfe_m1_fb
IOU 0.18235766244660132
Loaded DLab_Vperfe_m2_fb
IOU 0.25649041047700427
Loaded DLab_Vperfe_m4_fb
IOU 0.39581744544167125
Loaded DLab_Vperfe_m8_fb
IOU 0.44429039939213066
Loaded DLab_Vperfe_m16_fb
IOU 0.5388698538183123
Loaded DLab_Vperfe_m32_fb
IOU 0.6126735413546928
Loaded DLab_Vperfe_m50_fb
IOU 0.6276882060092092
